# 08 — Nested rolling-origin evaluation

**Purpose.** Freeze the pipeline and re-run it over several outer test years. For each outer year the
models see only earlier days: the Optuna search (fixed budget, declared in advance), the choice
between the full and the reduced predictor set, the scalers and the baseline climatology are all
fitted inside that training window, and the outer year is scored once.

**Answers:** Reviewer 4 comments 1, 2 (market-compatible origin variant), 5 and 7 (several outer test
sets); Reviewer 3 comment 5.

**Inputs:** `revision/df_final_features_causal.feather` (notebook 07).
**Outputs:** `revision/rolling_origin_predictions.csv`, `rolling_origin_params.json`,
`tables/S10_rolling_origin_point_metrics.csv`. The notebook resumes where it stopped, so a dropped
Colab session costs only the current year.
**Runtime:** ~40–60 min per outer year on a Colab GPU with the default budgets.

In [1]:
# --- Setup: Drive, paths, shared helpers -------------------------------------------------------
import os, sys, json, time, warnings, subprocess
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'shap', 'scienceplots',
                    'ephem', 'catboost', 'lightgbm'], check=False)
except Exception:
    pass
_BASE = os.environ.get('SARKOY_BASE', '/content/drive/MyDrive/windforecast_rev1/')
sys.path.insert(0, os.path.join(_BASE, 'revision'))
if not os.path.exists(os.path.join(_BASE, 'revision', 'revision_utils.py')):
    raise FileNotFoundError('Copy revision_utils.py into ' + os.path.join(_BASE, 'revision') + ' first.')
import revision_utils as ru
import numpy as np, pandas as pd, matplotlib.pyplot as plt
BASE, REV, IN_COLAB, FAST = ru.get_paths()
ru.set_style()
RES = ru.Results(REV)
RES.path = REV + 'results_08.json'
GPU = ru.has_gpu()
try:
    from IPython.display import display
except Exception:
    display = print
print('BASE:', BASE, '| REV:', REV, '| GPU:', GPU, '| FAST (smoke test):', FAST)
print(ru.log_versions(REV))


Mounted at /content/drive
BASE: /content/drive/MyDrive/windforecast_rev1/ | REV: /content/drive/MyDrive/windforecast_rev1/revision/ | GPU: True | FAST (smoke test): False
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'xgboost': '3.4.1', 'lightgbm': '4.6.0', 'catboost': '1.2.10', 'optuna': '5.0.0', 'shap': '0.52.0', 'statsmodels': '0.15.0', 'matplotlib': '3.10.0', 'seaborn': '0.13.2', 'pyarrow': '23.0.1', 'ephem': '4.2.1', 'tensorflow': '2.20.0', 'gpu': True}


In [2]:
# --- Configuration (declared before looking at any outer year) ----------------------------------
OUTER_YEARS = [2020, 2021, 2022, 2023, 2024]
MODELS = ['ElasticNetCV', 'SVR', 'RandomForest', 'LightGBM', 'CatBoost', 'XGBoost']
TRIALS = {'XGBoost': 50, 'LightGBM': 30, 'CatBoost': 15, 'RandomForest': 15, 'SVR': 20}
SPLITS = {'XGBoost': 5, 'LightGBM': 5, 'CatBoost': 3, 'RandomForest': 3, 'SVR': 5}
RESUME = True
if FAST:
    OUTER_YEARS, TRIALS, SPLITS = [2023, 2024], {k: 2 for k in TRIALS}, {k: 2 for k in SPLITS}
df = ru.load_matrix(BASE, REV, 'causal')
if (df.index.year == 2025).sum() >= 300:          # a genuinely unseen year, if the records reach it
    OUTER_YEARS = OUTER_YEARS + [2025]
X = ru.get_X(df); y = df[ru.TARGET]; ok = df['target_observed'].astype(bool)
print('outer years:', OUTER_YEARS, '| trials:', TRIALS, '| inner folds:', SPLITS)


Loaded causal matrix: (2964, 564), 2016-11-19 -> 2024-12-30, observed targets: 2957
outer years: [2020, 2021, 2022, 2023, 2024] | trials: {'XGBoost': 50, 'LightGBM': 30, 'CatBoost': 15, 'RandomForest': 15, 'SVR': 20} | inner folds: {'XGBoost': 5, 'LightGBM': 5, 'CatBoost': 3, 'RandomForest': 3, 'SVR': 5}


In [3]:
# --- Tuning helpers -----------------------------------------------------------------------------
from sklearn.model_selection import TimeSeriesSplit
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def inner_folds(Xtr, n_splits):
    return list(TimeSeriesSplit(n_splits=n_splits).split(Xtr))

def cv_score(name, params, Xtr, ytr, folds, cols=None):
    s = []
    for a, b in folds:
        cc = cols if cols is not None else Xtr.columns
        m = ru.make_model(name, params, gpu=GPU, fast=FAST).fit(Xtr.iloc[a][cc], ytr.iloc[a])
        s.append(ru.rmse(ytr.iloc[b], m.predict(Xtr.iloc[b][cc])))
    return float(np.mean(s))

def tune(name, Xtr, ytr, n_trials, n_splits):
    folds = inner_folds(Xtr, n_splits)
    def objective(trial):
        return cv_score(name, ru.suggest(trial, name), Xtr, ytr, folds)
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, float(study.best_value), folds

def reduced_cv(params, Xtr, ytr, folds):
    """Section 2.4 reduction repeated inside every inner fold, so the choice never sees the outer year."""
    s, sizes = [], []
    for a, b in folds:
        red = ru.reduce_features(Xtr.iloc[a], ytr.iloc[a], n_trees=300, fast=FAST)
        m = ru.make_model('XGBoost', params, gpu=GPU, fast=FAST).fit(Xtr.iloc[a][red], ytr.iloc[a])
        s.append(ru.rmse(ytr.iloc[b], m.predict(Xtr.iloc[b][red]))); sizes.append(len(red))
    return float(np.mean(s)), int(np.mean(sizes))


In [4]:
# --- Outer loop ----------------------------------------------------------------------------------
pred_file, par_file = REV + 'rolling_origin_predictions.csv', REV + 'rolling_origin_params.json'
done = set()
if RESUME and os.path.exists(pred_file):
    done = set(pd.read_csv(pred_file, usecols=['outer_year']).outer_year.unique())
params_all = json.load(open(par_file)) if (RESUME and os.path.exists(par_file)) else {}
Xs = X.shift(1)                                   # market-compatible origin: predictors up to day t−1

for Y in OUTER_YEARS:
    if Y in done:
        print('already done:', Y); continue
    t_year = time.time()
    trm = ok & (df.index.year < Y); tem = ok & (df.index.year == Y)
    if tem.sum() < 30 or trm.sum() < 300:
        print('skipping', Y, '- not enough data'); continue
    Xtr, ytr, Xte, yte = X[trm], y[trm], X[tem], y[tem]
    base = ru.baseline_predictions(df, trm)
    out = pd.DataFrame({'date': Xte.index, 'outer_year': Y, 'y_true': yte.to_numpy()})
    for b in ('persistence', 'seasonal_naive', 'moving_average_7d', 'persistence_2d'):
        out[b] = base.loc[tem, b].to_numpy()
    info = {}
    for name in MODELS:
        t0 = time.time()
        if name == 'ElasticNetCV':
            bp, cvs = {}, None
            m = ru.make_model('ElasticNetCV', fast=FAST).fit(Xtr, ytr)
        else:
            bp, cvs, folds = tune(name, Xtr, ytr, TRIALS[name], SPLITS[name])
            m = ru.make_model(name, bp, gpu=GPU, fast=FAST).fit(Xtr, ytr)
        out[name] = m.predict(Xte)
        info[name] = dict(best_params=bp, inner_cv_rmse=cvs, seconds=round(time.time() - t0, 1),
                          n_trials=TRIALS.get(name), inner_splits=SPLITS.get(name))
        print(f'  {Y} {name:14s} inner CV {cvs if cvs is None else round(cvs, 4)} '
              f'| outer RMSE {ru.rmse(yte, out[name]):.4f} | {time.time() - t0:.0f} s')
        if name == 'XGBoost':
            cv_red, n_red = reduced_cv(bp, Xtr, ytr, folds)
            use_red = cv_red < cvs
            red = ru.reduce_features(Xtr, ytr, fast=FAST)
            mr = ru.make_model('XGBoost', bp, gpu=GPU, fast=FAST).fit(Xtr[red], ytr)
            out['XGBoost_reduced'] = mr.predict(Xte[red])
            out['XGBoost_selected'] = out['XGBoost_reduced'] if use_red else out['XGBoost']
            trs = trm & Xs.notna().all(axis=1)
            ms = ru.make_model('XGBoost', bp, gpu=GPU, fast=FAST).fit(Xs[trs], y[trs])
            out['XGBoost_origin_t-1'] = ms.predict(Xs[tem])
            info[name].update(inner_cv_rmse_reduced=cv_red, use_reduced_set=bool(use_red),
                              n_reduced_inner=n_red, n_reduced_final=len(red), reduced_features=red)
            print(f'  {Y} feature set chosen by inner CV: '
                  f"{'reduced (' + str(len(red)) + ')' if use_red else 'full (559)'}"
                  f' | full {cvs:.4f} vs reduced {cv_red:.4f}')
    out.to_csv(pred_file, mode='a', header=not os.path.exists(pred_file), index=False)
    params_all[str(Y)] = info
    json.dump(params_all, open(par_file, 'w'), indent=1, default=str)
    print(f'{Y} finished in {(time.time() - t_year) / 60:.1f} min')


already done: 2020
already done: 2021
already done: 2022
  2023 ElasticNetCV   inner CV None | outer RMSE 1.1470 | 70 s
  2023 SVR            inner CV 1.1669 | outer RMSE 1.0698 | 47 s
  2023 RandomForest   inner CV 1.0296 | outer RMSE 1.0449 | 842 s
  2023 LightGBM       inner CV 1.0467 | outer RMSE 1.0167 | 542 s
  2023 CatBoost       inner CV 1.027 | outer RMSE 1.0469 | 241 s
  2023 XGBoost        inner CV 1.0318 | outer RMSE 1.0274 | 632 s
  2023 feature set chosen by inner CV: full (559) | full 1.0318 vs reduced 1.0980
2023 finished in 45.2 min
  2024 ElasticNetCV   inner CV None | outer RMSE 1.0650 | 78 s
  2024 SVR            inner CV 1.1435 | outer RMSE 1.0688 | 62 s
  2024 RandomForest   inner CV 1.0468 | outer RMSE 0.9608 | 1021 s
  2024 LightGBM       inner CV 1.0228 | outer RMSE 0.9532 | 838 s
  2024 CatBoost       inner CV 1.0334 | outer RMSE 0.9514 | 229 s
  2024 XGBoost        inner CV 1.0152 | outer RMSE 0.9531 | 818 s
  2024 feature set chosen by inner CV: full (559) |

In [5]:
# --- Point metrics per outer year -----------------------------------------------------------------
R = pd.read_csv(pred_file, parse_dates=['date'])
cols = [c for c in R.columns if c not in ('date', 'outer_year', 'y_true')]
rows = []
for Y, g in R.groupby('outer_year'):
    for c in cols:
        if g[c].notna().all():
            rows.append(dict(outer_year=Y, model=c, **ru.metrics(g.y_true, g[c])))
M = pd.DataFrame(rows)
M.to_csv(REV + 'tables/S10_rolling_origin_point_metrics.csv', index=False)
display(M.pivot(index='model', columns='outer_year', values='RMSE').round(4))
n_red = sum(1 for v in params_all.values() if v.get('XGBoost', {}).get('use_reduced_set'))
RES.put('RO_YEARS', f"{min(R.outer_year)}–{max(R.outer_year)}").put('RO_NYEARS', R.outer_year.nunique(), 'd')
RES.put('RO_N_REDUCED_YEARS', n_red, 'd').save()


outer_year,2020,2021,2022,2023,2024
model,,,,,
CatBoost,1.0833,1.0234,0.9939,1.0469,0.9514
ElasticNetCV,1.2033,1.1226,1.1045,1.1470,1.0650
LightGBM,1.0895,1.0375,0.9885,1.0167,0.9532
RandomForest,1.0783,1.0427,1.0136,1.0449,0.9608
SVR,1.2433,1.0647,1.0311,1.0698,1.0688
XGBoost,1.0690,1.0197,0.9863,1.0274,0.9531
XGBoost_origin_t-1,1.4484,1.3365,1.3589,1.3429,1.2103
XGBoost_reduced,1.0897,1.1031,1.0925,1.0944,1.0164
XGBoost_selected,1.0690,1.0197,0.9863,1.0274,0.9531


3 values written to /content/drive/MyDrive/windforecast_rev1/revision/results_08.json


### Disconnect Runtime
Wait 20 seconds and then disconnect to save compute resources.

In [6]:
import time
from google.colab import runtime

print("Execution complete. Waiting 20 seconds before disconnecting...")
time.sleep(20)
print("Disconnecting runtime now.")
runtime.unassign()


Execution complete. Waiting 20 seconds before disconnecting...
Disconnecting runtime now.
